In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [1]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor
from safetensors import safe_open
import safetensors.torch as st
import gc
import json

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Name: NVIDIA A100 80GB PCIe
VRAM: 79.1 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "google/gemma-4-E4B-it"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound-e4b"
LOCAL_PATH = "./local_model-e4b"

In [ ]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

In [6]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [7]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

In [8]:
layer_config = {}
for name, _ in model.named_modules():
    if name.startswith(("model.vision_tower", "model.audio_tower",
                        "model.multi_modal_projector", "model.audio_projector")):
        layer_config[name] = {"bits": 32}

print(f"Skipping {len(layer_config)} non-LM modules")


Skipping 889 non-LM modules


In [9]:
TUNING_CONFIG = {
    "group_size": 128,
    "sym": True,
    "iters": 0,               # RTN mode — required for Gemma 4
    "disable_opt_rtn": True,
    "nsamples": 256,
    "seqlen": 2048,
    "low_gpu_mem_usage": False,
    "quant_nontext_module": False,
    "layer_config": layer_config,
}

In [10]:
def push_to_hub(local_dir, repo_name, token):
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")
    try:
        api = HfApi()
        create_repo(full_repo_id, repo_type="model", exist_ok=True, private=False, token=token)
        api.upload_folder(folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token)
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [11]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-04-12 07:56:39 INFO autoround.py L178: using MLLM mode for multimodal model.
2026-04-12 07:56:39 INFO base.py L517: using torch.bfloat16 for quantization tuning


In [12]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq", inplace=True
)

2026-04-12 07:56:40 WARNING formats.py L166: some layers are skipped quantization (shape not divisible by 32): 
2026-04-12 07:56:40 WARNING modeling_utils.py L4435: `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-04-12 07:56:42 WARNING utils.py L445: 'model.vision_tower.encoder.rotary_emb' exists in the model but is not a supported quantization target in the current scheme, ignoring its setting in `layer_config`
2026-04-12 07:56:42 WARNING utils.py L445: 'model.vision_tower.encoder.layers.0.self_attn.q_norm' exists in the model but is not a supported quantization target in the current scheme, ignoring its setting in `layer_config`
2026-04-12 07:56:42 WARNING utils.py L445: 'model.vision_tower.encoder.layers.0.self_attn.k_norm' exists in the model but is not a supported quantization target in the current scheme, ignoring its setting in `layer_config`
2026-04-12 07:56:42 WARNING utils.py L445: 'model.vision_tower.encoder.laye

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

packing model.language_model.layers.41.per_layer_projection: 100%|██████████| 589/589 [00:03<00:00, 173.27it/s]          


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-12 07:58:56 INFO device.py L1692: 'peak_ram': 1.45GB, 'peak_vram': 15.21GB


(Gemma4ForConditionalGeneration(
   (model): Gemma4Model(
     (language_model): Gemma4TextModel(
       (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
       (layers): ModuleList(
         (0-23): 24 x Gemma4TextDecoderLayer(
           (self_attn): Gemma4TextAttention(
             (q_proj): QuantLinear()
             (q_norm): Gemma4RMSNorm()
             (k_norm): Gemma4RMSNorm()
             (v_norm): Gemma4RMSNorm()
             (k_proj): QuantLinear()
             (v_proj): QuantLinear()
             (o_proj): QuantLinear()
           )
           (mlp): Gemma4TextMLP(
             (gate_proj): QuantLinear()
             (up_proj): QuantLinear()
             (down_proj): QuantLinear()
             (act_fn): GELUTanh()
           )
           (input_layernorm): Gemma4RMSNorm()
           (post_attention_layernorm): Gemma4RMSNorm()
           (pre_feedforward_layernorm): Gemma4RMSNorm()
           (post_feedforward_layernorm): Gemma4RMSNorm()
          

In [20]:
def fix_softcap_tensors(model_dir):
    print(f"\n[Fix] Scanning {model_dir}...")
    for fname in sorted(os.listdir(model_dir)):
        if not fname.endswith(".safetensors"):
            continue
        fpath = os.path.join(model_dir, fname)
        with safe_open(fpath, framework="pt", device="cpu") as f:
            all_keys = list(f.keys())
            bad_keys = [k for k in all_keys if "softcap" in k]
            if not bad_keys:
                print(f"  {fname}: clean")
                continue
            good_tensors = {k: f.get_tensor(k) for k in all_keys if k not in bad_keys}
        st.save_file(good_tensors, fpath)
        del good_tensors
        gc.collect()
        print(f"  {fname}: removed {len(bad_keys)} softcap entries → {bad_keys}")

    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.exists(idx_path):
        with open(idx_path) as f:
            idx = json.load(f)
        bad_idx = [k for k in list(idx["weight_map"].keys()) if "softcap" in k]
        for k in bad_idx:
            del idx["weight_map"][k]
        with open(idx_path, "w") as f:
            json.dump(idx, f, indent=2)
        print(f"  index.json: removed {len(bad_idx)} entries")
    print("[Fix] ✅ Done")



In [21]:
fix_softcap_tensors(os.path.join(OUTPUT_BASE_DIR, "auto-round-auto-gptq"))
fix_softcap_tensors(os.path.join(OUTPUT_BASE_DIR, "auto-gptq"))


[Fix] Scanning ./AutoRound-e4b/auto-round-auto-gptq...
  model-00001-of-00002.safetensors: clean
  model-00002-of-00002.safetensors: clean
  index.json: removed 12 entries
[Fix] ✅ Done

[Fix] Scanning ./AutoRound-e4b/auto-gptq...
  model-00001-of-00002.safetensors: clean
  model-00002-of-00002.safetensors: removed 12 softcap entries → ['model.audio_tower.layers.0.self_attn.softcap', 'model.audio_tower.layers.1.self_attn.softcap', 'model.audio_tower.layers.10.self_attn.softcap', 'model.audio_tower.layers.11.self_attn.softcap', 'model.audio_tower.layers.2.self_attn.softcap', 'model.audio_tower.layers.3.self_attn.softcap', 'model.audio_tower.layers.4.self_attn.softcap', 'model.audio_tower.layers.5.self_attn.softcap', 'model.audio_tower.layers.6.self_attn.softcap', 'model.audio_tower.layers.7.self_attn.softcap', 'model.audio_tower.layers.8.self_attn.softcap', 'model.audio_tower.layers.9.self_attn.softcap']
  index.json: removed 12 entries
[Fix] ✅ Done


In [6]:
def copy_processor_configs(model_id, output_dir):
    for filename in ["preprocessor_config.json", "processor_config.json", "chat_template.jinja"]:
        try:
            shutil.copy(hf_hub_download(model_id, filename), output_dir)
            print(f"  Copied {filename} → {output_dir}")
        except Exception:
            pass

In [7]:
copy_processor_configs(MODEL_ID, os.path.join(OUTPUT_BASE_DIR, "auto-round-auto-gptq"))
copy_processor_configs(MODEL_ID, os.path.join(OUTPUT_BASE_DIR, "auto-gptq"))

In [8]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [11]:
if hf_token:
    # Push auto_round format
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token
    )
    # Push auto_gptq format (vLLM compatible)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "auto-gptq"),
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")



[Hub] Pushing ./AutoRound-e4b/auto-round-auto-gptq to Vishva007/gemma-4-E4B-it-W4A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-E4B-it-W4A16-AutoRound

[Hub] Pushing ./AutoRound-e4b/auto-gptq to Vishva007/gemma-4-E4B-it-W4A16-AutoRound-GPTQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-E4B-it-W4A16-AutoRound-GPTQ
